
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import json
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017A2E81E510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017A2E81EF90>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### Lets build a recipe based system

In [3]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    name:str=Field(description="The name of the dish")
    cuisine:str=Field(description="The cuisine style of the dish")
    prep_time:int=Field(description="Preparation time in minutes")
    ingredients:list[str]=Field(description="List of ingredients needed for the dish")

In [4]:
# not an model invocation, with_structured_output(Recipe) just a runnable
model_with_structure=model.with_structured_output(Recipe)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017A2E81E510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017A2E81EF90>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Recipe', 'description': '', 'parameters': {'properties': {'name'

In [5]:
model.invoke("Tell me about khachapuri, the Georgian cheese-filled bread")

AIMessage(content='**Khachapuri – the Georgian “cheese‑filled bread” that melts in your mouth**\n\nKhachapuri is more than just a pastry; it’s a national treasure, a culinary emblem of Georgia, and a dish that invites everyone to sit down, share a meal, and experience the country’s warm hospitality. Below is a deep dive into what makes khachapuri special, its history, regional variations, how it’s traditionally baked, and a quick recipe to bring a slice of Georgia into your kitchen.\n\n---\n\n## 1. What is Khachapuri?\n\n- **Name origin**: The word *khachapuri* comes from the Georgian *khachap* (cheese) + *uri* (bread). In many Georgian dialects, it’s pronounced “khachapuri” or “khashapuri.”\n- **Basic concept**: A yeasted bread dough filled (or topped) with cheese, sometimes with egg, butter, or other ingredients. The dough is baked until golden, and the cheese inside is gooey and melty.\n- **Why it’s special**: The combination of a soft, slightly chewy crust with the savory, salty ch

In [6]:
response=model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")

print(json.dumps(response.model_dump(),indent=2, default=str))

{
  "name": "Khachapuri",
  "cuisine": "Georgian",
  "prep_time": 90,
  "ingredients": [
    "500g all-purpose flour",
    "1 tsp salt",
    "1 tsp sugar",
    "1 tsp instant yeast",
    "300ml warm water",
    "2 tbsp olive oil",
    "200g cheese (a mix of feta and mozzarella or traditional sulguni)",
    "1 egg (for egg wash)"
  ]
}


### MEssage output alongside parsed structure

In [7]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    """A recipe with details."""
    name: str = Field(..., description="The name of the dish")
    cuisine: str = Field(..., description="The cuisine style of the dish")
    prep_time: int = Field(..., description="Preparation time in minutes")
    ingredients: list[str] = Field(..., description="List of ingredients needed for the dish")

model_with_structure = model.with_structured_output(Recipe, include_raw=True)  

response = model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")


In [8]:
response
# print(json.dumps(
#     {
#         "raw": response["raw"].model_dump(),
#         "parsed": response["parsed"].model_dump(),
#         "parsing_error": response["parsing_error"],
#     },
#     indent=2,
#     default=str,
#     ensure_ascii=False,
# ))

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for recipe details for khachapuri, Georgian cheese-filled bread. The system says we can use a function call to get a recipe. The function signature: Recipe({cuisine, ingredients, name, prep_time}). We need to supply these arguments. Provide the recipe details. We should call the function.', 'tool_calls': [{'id': 'fc_d2f9816f-3e5b-47c5-93b8-88ef43e164fb', 'function': {'arguments': '{"cuisine":"Georgian","ingredients":["500g all-purpose flour","10g active dry yeast","1 tsp sugar","1 tsp salt","250ml warm milk","2 tbsp olive oil","200g cheese blend (e.g., sulguni, feta, mozzarella)","1 egg (for egg wash)","Butter (optional)"],"name":"Khachapuri","prep_time":60}', 'name': 'Recipe'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 167, 'prompt_tokens': 173, 'total_tokens': 340, 'completion_time': 0.180429163, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time'

### Nested Structure

In [9]:
# This may fail if the model did not generated the output according to our pydantic model
from pydantic import BaseModel, Field


class Ingredient(BaseModel):
    item: str
    quantity: str

class RecipeDetails(BaseModel):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for khachapuri, the Georgian cheese-filled bread")
response

RecipeDetails(name='Khachapuri', cuisine='Georgian', ingredients=[Ingredient(item='All-purpose flour', quantity='500g'), Ingredient(item='Warm water', quantity='300ml'), Ingredient(item='Dry yeast', quantity='7g (1 packet)'), Ingredient(item='Sugar', quantity='10g'), Ingredient(item='Salt', quantity='10g'), Ingredient(item='Olive oil', quantity='30ml'), Ingredient(item='Butter', quantity='50g'), Ingredient(item='Cheese (a mix of sulguni and feta or mozzarella)', quantity='300g'), Ingredient(item='Egg yolk', quantity='1 (for egg wash)')], steps=['Preheat the oven to 220°C (425°F).', 'In a small bowl, dissolve the yeast and sugar in warm water. Let it sit for 5-10 minutes until frothy.', 'In a large mixing bowl, combine the flour and salt. Add the yeast mixture, olive oil, and a little warm water if needed. Knead the dough until it is smooth and elastic, about 8-10 minutes. If the dough is too sticky, add a little more flour.', 'Place the dough in a lightly oiled bowl, cover with a damp 

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [10]:
from typing import Annotated

from typing_extensions import TypedDict


class RecipeDict(TypedDict):
    """A recipe with details."""
    name: Annotated[str, ..., "The name of the dish"]
    cuisine: Annotated[str, ..., "The cuisine style of the dish"]
    prep_time: Annotated[int, ..., "Preparation time in minutes"]
    ingredients: Annotated[list[str], ..., "List of ingredients needed for the dish"]


model_withtypedict=model.with_structured_output(RecipeDict)
response=model_withtypedict.invoke("Please provide the recipe details for pizza")
response

{'cuisine': 'Italian',
 'ingredients': ['Pizza dough',
  'Tomato sauce',
  'Mozzarella cheese',
  'Olive oil',
  'Basil leaves',
  'Salt',
  'Pepper'],
 'name': 'Classic Margherita Pizza',
 'prep_time': 60}

In [11]:
bad = RecipeDict(
    name=123,                 # should be str
    cuisine=None,             # should be str
    prep_time="not a number", # should be int
    ingredients="cheese",     # should be list[str]
)
print(bad)   # no error

{'name': 123, 'cuisine': None, 'prep_time': 'not a number', 'ingredients': 'cheese'}


In [12]:
class Ingredient(TypedDict):  # noqa: F811
    item: str
    quantity: str

class RecipeDetails(TypedDict):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for pancakes")
response

{'cuisine': 'American',
 'ingredients': [{'item': 'All-purpose flour', 'quantity': '1 1/2 cups'},
  {'item': 'Baking powder', 'quantity': '3 1/2 teaspoons'},
  {'item': 'Salt', 'quantity': '1 teaspoon'},
  {'item': 'White sugar', 'quantity': '1 tablespoon'},
  {'item': 'Milk', 'quantity': '1 1/4 cups'},
  {'item': 'Egg', 'quantity': '1'},
  {'item': 'Melted butter', 'quantity': '3 tablespoons'}],
 'name': 'Classic Pancakes',
 'prep_time': 20,
 'steps': ['In a large bowl, sift together the flour, baking powder, salt, and sugar.',
  'Make a well in the center and pour in the milk, egg, and melted butter; mix until smooth.',
  'Heat a lightly oiled griddle or frying pan over medium-high heat.',
  'Pour or scoop the batter onto the griddle, using approximately 1/4 cup for each pancake.',
  'Brown on both sides and serve hot with butter and syrup.']}

In [13]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2025-08-05',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [14]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")

In [15]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029', additional_kwargs={}, response_metadata={}, id='cc96ea09-609a-4206-90e5-6da9d2dd0951'),
  AIMessage(content='{"name":"Elena Marquez","email":"elena.marquez@brightleaf-consulting.com","phone":"(312) 774-5029"}', additional_kwargs={'reasoning_content': 'We need to produce JSON following the schema ContactInfo. Must include all required fields: name, email, phone. Provide compact JSON. Ensure valid JSON. The input: "Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029". So name: "Elena Marquez", email: "elena.marquez@brightleaf-consulting.com", phone: "(312) 774-5029". Output JSON. Ensure no trailing spaces. Use compact: no line breaks? But the response format likely expects JSON object. We can output as single line or multiline but it\'s JSON. Use compact: no unnecessary whitespace. Let\'s output:\n\n{"name":"Elena Marquez","email":"ele

In [16]:
result["structured_response"]

ContactInfo(name='Elena Marquez', email='elena.marquez@brightleaf-consulting.com', phone='(312) 774-5029')

In [17]:
## Dataclass

from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result["structured_response"]

ContactInfo(name='Elena Marquez', email='elena.marquez@brightleaf-consulting.com', phone='(312) 774-5029')

### dataclass and pydantic in same example

In [18]:
from dataclasses import dataclass

from pydantic import BaseModel, ValidationError


@dataclass
class ContactDC:
    name: str
    email: str
    phone: str

class ContactPD(BaseModel):
    name: str
    email: str
    phone: str

print(ContactDC(name=123, email=None, phone=["x"]))   # works, no error

try:
    ContactPD(name=123, email=None, phone=["x"])
except ValidationError as e:
    print(e)                                           # raises

ContactDC(name=123, email=None, phone=['x'])
3 validation errors for ContactPD
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
email
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
phone
  Input should be a valid string [type=string_type, input_value=['x'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
